# 01 · Coleta

Baixa as internações do SIH/SUS (arquivo RD) do RN para os 12 meses de 2024, direto do FTP do DATASUS, e os nomes dos municípios pela API do IBGE.

**Escopo:** internações eletivas e de urgência (caráter da internação), que formam as duas camadas da rede.

**Saídas**
- `data/processed/rd_rn_2024.parquet`: uma linha por internação, só as colunas usadas no projeto, com a camada (`eletiva` ou `urgencia`)
- `data/processed/municipios_rn.csv`: código IBGE de 6 dígitos e nome
- `data/processed/coordenadas_rn.csv`: latitude e longitude da sede de cada município ([kelvins/municipios-brasileiros](https://github.com/kelvins/municipios-brasileiros))
- `data/processed/distancias_rn.csv`: distância e tempo por estrada entre as sedes, pelo [OSRM](https://project-osrm.org) sobre o OpenStreetMap, e distância em linha reta
- `data/processed/hospitais_rn.csv`: nome de cada hospital (CNES) que aparece nas internações, pela [API de dados abertos do Ministério da Saúde](https://apidadosabertos.saude.gov.br)

**Dependências:** `pandas`, `pyarrow`, `pyreaddbc`, `dbfread`


In [1]:
from ftplib import FTP
from pathlib import Path

import pandas as pd
from dbfread import DBF
from pyreaddbc import dbc2dbf

UF = "RN"
UF_IBGE = 24
ANO = 2024
MESES = range(1, 13)

FTP_HOST = "ftp.datasus.gov.br"
FTP_DIR = "/dissemin/publicos/SIHSUS/200801_/Dados"

COLUNAS = [
    "ANO_CMPT", "MES_CMPT",    # competência
    "MUNIC_RES", "MUNIC_MOV",  # origem e destino do fluxo
    "CNES",                    # hospital
    "DIAG_PRINC",              # CID-10
    "CAR_INT",                 # caráter da internação (eletiva, urgência...)
    "IDENT",                   # tipo de AIH: 1 = normal, 5 = continuação
    "DT_INTER",                # data da internação
]

RAW = Path("data/raw")
PROCESSED = Path("data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. Download

In [2]:
def nome_arquivo(mes: int) -> str:
    return f"RD{UF}{ANO % 100:02d}{mes:02d}.dbc"

with FTP(FTP_HOST) as ftp:
    ftp.login()
    ftp.cwd(FTP_DIR)
    for mes in MESES:
        destino = RAW / nome_arquivo(mes)
        if destino.exists():
            continue
        parcial = destino.with_suffix(".part")
        with open(parcial, "wb") as f:
            ftp.retrbinary(f"RETR {destino.name}", f.write)
        parcial.rename(destino)
        print("baixado:", destino.name)

## 2. Leitura

`.dbc` é um `.dbf` comprimido. O `pyreaddbc` descomprime para `.dbf` e o `dbfread` lê os registros.

In [3]:
def ler_rd(dbc: Path) -> pd.DataFrame:
    dbf = dbc.with_suffix(".dbf")
    if not dbf.exists():
        dbc2dbf(str(dbc), str(dbf))
    registros = DBF(dbf, encoding="iso-8859-1", load=False)
    return pd.DataFrame({c: r[c] for c in COLUNAS} for r in registros)

df = pd.concat([ler_rd(RAW / nome_arquivo(m)) for m in MESES], ignore_index=True)
df.shape

(216884, 9)

## 3. Tipos

In [4]:
for col in ["MUNIC_RES", "MUNIC_MOV", "CNES", "DIAG_PRINC", "CAR_INT", "IDENT"]:
    df[col] = df[col].astype(str).str.strip()

df["ANO_CMPT"] = df["ANO_CMPT"].astype(int)
df["MES_CMPT"] = df["MES_CMPT"].astype(int)
df["DT_INTER"] = pd.to_datetime(df["DT_INTER"], format="%Y%m%d")

## 4. Verificação

In [5]:
assert (df["ANO_CMPT"] == ANO).all()
assert sorted(df["MES_CMPT"].unique()) == list(MESES)

codigo_valido = r"\d{6}"
print("MUNIC_RES inválidos:", (~df["MUNIC_RES"].str.fullmatch(codigo_valido)).sum())
print("MUNIC_MOV inválidos:", (~df["MUNIC_MOV"].str.fullmatch(codigo_valido)).sum())

print("Tipo de AIH:", df["IDENT"].value_counts().to_dict())
print(f"Internações com entrada em {ANO}: {(df['DT_INTER'].dt.year == ANO).mean():.1%}")

df.groupby("MES_CMPT").size()

MUNIC_RES inválidos: 0
MUNIC_MOV inválidos: 0
Tipo de AIH: {'1': 215993, '5': 891}
Internações com entrada em 2024: 93.9%


MES_CMPT
1     17493
2     17164
3     17878
4     17727
5     17920
6     18716
7     19293
8     19768
9     18797
10    19005
11    17887
12    15236
dtype: int64

## 5. Caráter da internação: eletiva e urgência

O projeto compara duas camadas: **eletiva** (internação agendada) e **urgência** (todo caráter não eletivo). Os códigos de `CAR_INT` vêm da tabela de conversão oficial do SIH, `CNV/CARATEND.CNV`, dentro de `TAB_SIH.zip` (`ftp.datasus.gov.br/dissemin/publicos/SIHSUS/200801_/Auxiliar/`), a mesma que o TabWin usa. O informe técnico `IT_SIHSUS_1603.pdf` (pasta `Doc/`) define a variável como "caráter da internação".

AIHs com código fora da tabela seriam descartadas.

In [6]:
CARATER = {
    "01": "Eletivo",
    "02": "Urgência",
    "03": "Acidente no local de trabalho ou a serviço da empresa",
    "04": "Acidente no trajeto para o trabalho",
    "05": "Outros tipos de acidente de trânsito",
    "06": "Outros tipos de lesões e envenenamentos por agentes químicos ou físicos",
}

fora_da_tabela = ~df["CAR_INT"].isin(CARATER)
print(f"Removidas (caráter fora da tabela): {fora_da_tabela.sum()}")
df = df[~fora_da_tabela].copy()

df["CAMADA"] = df["CAR_INT"].eq("01").map({True: "eletiva", False: "urgencia"})

df.groupby(["CAMADA", "CAR_INT"]).size().rename("internações").rename(index=CARATER, level="CAR_INT")

Removidas (caráter fora da tabela): 0


CAMADA    CAR_INT                                                                
eletiva   Eletivo                                                                     80986
urgencia  Urgência                                                                   124524
          Outros tipos de acidente de trânsito                                         4005
          Outros tipos de lesões e envenenamentos por agentes químicos ou físicos      7369
Name: internações, dtype: int64

## 6. Municípios do RN (IBGE)

A API do IBGE usa códigos de 7 dígitos; o DATASUS usa os 6 primeiros.

In [7]:
url = f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{UF_IBGE}/municipios"
municipios = pd.read_json(url)[["id", "nome"]]
municipios["cod6"] = municipios["id"].astype(str).str[:6]
municipios = municipios[["cod6", "nome"]]

assert len(municipios) == 167
municipios.head()

,cod6,nome
0,240010,Acari
1,240020,Assú
2,240030,Afonso Bezerra
3,240040,Água Nova
4,240050,Alexandria


In [8]:
nomes = municipios.set_index("cod6")["nome"]

df["NOME_RES"] = df["MUNIC_RES"].map(nomes)
df["NOME_MOV"] = df["MUNIC_MOV"].map(nomes)

assert df["NOME_MOV"].notna().all() 
print(f"Residentes fora do RN (sem nome): {df['NOME_RES'].isna().mean():.1%}")

Residentes fora do RN (sem nome): 0.9%


## 7. Salvar

In [9]:
df.to_parquet(PROCESSED / f"rd_{UF.lower()}_{ANO}.parquet", index=False)
municipios.to_csv(PROCESSED / "municipios_rn.csv", index=False)

In [10]:
df = pd.read_parquet(PROCESSED / f"rd_{UF.lower()}_{ANO}.parquet")

In [11]:
df

,ANO_CMPT,MES_CMPT,MUNIC_RES,MUNIC_MOV,CNES,DIAG_PRINC,CAR_INT,IDENT,DT_INTER,CAMADA,NOME_RES,NOME_MOV
0,2024,1,240310,240310,2476487,O820,01,1,2024-01-13,eletiva,Currais Novos,Currais Novos
1,2024,1,240310,240310,2476487,O458,02,1,2024-01-13,urgencia,Currais Novos,Currais Novos
2,2024,1,241450,240800,2503689,I602,02,1,2023-12-18,urgencia,Umarizal,Mossoró
3,2024,1,240990,240800,2503689,I64,02,1,2023-12-07,urgencia,Pendências,Mossoró
4,2024,1,240020,240800,2503689,K800,02,1,2023-12-07,urgencia,Assú,Mossoró
...,...,...,...,...,...,...,...,...,...,...,...,...
216879,2024,12,241460,241460,2407973,N179,01,1,2024-12-20,eletiva,Upanema,Upanema
216880,2024,12,241460,241460,2407973,J189,01,1,2024-12-02,eletiva,Upanema,Upanema
216881,2024,12,241460,241460,2407973,N188,01,1,2024-12-25,eletiva,Upanema,Upanema
216882,2024,12,241475,241475,2409259,A09,02,1,2024-11-17,urgencia,Venha-Ver,Venha-Ver


## 8. Coordenadas das sedes

O SIH não traz localização. As coordenadas vêm da base pública [kelvins/municipios-brasileiros](https://github.com/kelvins/municipios-brasileiros), que usa o código IBGE de 7 dígitos.

In [12]:
COORD_URL = "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv"

coord = pd.read_csv(COORD_URL)
coord = coord[coord["codigo_uf"] == UF_IBGE]
coord["cod6"] = coord["codigo_ibge"].astype(str).str[:6]
coord = coord.set_index("cod6")[["latitude", "longitude"]]

assert set(coord.index) == set(municipios["cod6"])
coord.to_csv(PROCESSED / "coordenadas_rn.csv")

## 9. Distâncias por estrada (OSRM)

O serviço `table` do OSRM devolve a distância (m) e o tempo (s) de carro entre todos os pares de pontos. O servidor público aceita no máximo 100 coordenadas por consulta, então a matriz 167 × 167 é pedida em blocos. O resultado fica salvo em disco: se o arquivo já existe, o serviço não é consultado de novo.

A distância em linha reta (haversine) fica ao lado, para comparação.

**Limitações:** a rota vai de sede a sede, não da casa do paciente ao hospital; o mapa do OpenStreetMap pode estar desatualizado em estradas do interior.

In [13]:
import json
import time
import urllib.request
from itertools import product

import numpy as np

OSRM = "https://router.project-osrm.org/table/v1/driving/"
BLOCO = 50  # 50 origens + 50 destinos = 100 coordenadas por consulta

def consultar_bloco(origens, destinos):
    pontos = list(origens) + list(destinos)
    coords = ";".join(f"{coord.longitude[c]},{coord.latitude[c]}" for c in pontos)
    fontes = ";".join(str(i) for i in range(len(origens)))
    alvos = ";".join(str(i) for i in range(len(origens), len(pontos)))
    url = f"{OSRM}{coords}?annotations=distance,duration&sources={fontes}&destinations={alvos}"
    with urllib.request.urlopen(url, timeout=60) as r:
        resp = json.load(r)
    assert resp["code"] == "Ok", resp
    return [(o, d, resp["distances"][i][j], resp["durations"][i][j])
            for i, o in enumerate(origens) for j, d in enumerate(destinos)]

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    h = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(h))

arquivo = PROCESSED / "distancias_rn.csv"
if not arquivo.exists():
    cods = list(coord.index)
    blocos = [cods[i:i + BLOCO] for i in range(0, len(cods), BLOCO)]
    linhas = []
    for origens, destinos in product(blocos, blocos):
        linhas += consultar_bloco(origens, destinos)
        time.sleep(1)  # uso educado do servidor público

    dist = pd.DataFrame(linhas, columns=["origem", "destino", "metros", "segundos"])
    dist = dist[dist["origem"] != dist["destino"]]
    o, d = coord.loc[dist["origem"]], coord.loc[dist["destino"]]
    dist["km_reta"] = haversine_km(o["latitude"].values, o["longitude"].values,
                                   d["latitude"].values, d["longitude"].values)
    dist["km_estrada"] = dist["metros"] / 1000
    dist["horas"] = dist["segundos"] / 3600
    dist[["origem", "destino", "km_reta", "km_estrada", "horas"]].round(2).to_csv(arquivo, index=False)

dist = pd.read_csv(arquivo, dtype={"origem": str, "destino": str})
print(f"Pares: {len(dist):,} (esperado {167 * 166:,})")
print(f"Sem rota: {dist['km_estrada'].isna().sum()}")

Pares: 27,722 (esperado 27,722)
Sem rota: 0


### Verificação: estrada contra linha reta

In [14]:
dist["razao"] = dist["km_estrada"] / dist["km_reta"]
print(dist["razao"].describe().round(2))

cod = municipios.set_index("nome")["cod6"]
pares = [("Natal", "Mossoró"), ("Caicó", "Natal"), ("Pau dos Ferros", "Mossoró"), ("Pau dos Ferros", "Natal")]
idx = dist.set_index(["origem", "destino"])
pd.DataFrame([{"origem": a, "destino": b, **idx.loc[(cod[a], cod[b]), ["km_reta", "km_estrada", "horas"]]}
              for a, b in pares])

count    27722.00
mean         1.36
std          0.22
min          1.00
25%          1.25
50%          1.31
75%          1.42
max         10.99
Name: razao, dtype: float64


,origem,destino,km_reta,km_estrada,horas
0,Natal,Mossoró,247.32,280.66,4.43
1,Caicó,Natal,223.39,273.27,3.69
2,Pau dos Ferros,Mossoró,139.84,152.10,1.85
3,Pau dos Ferros,Natal,334.59,429.72,5.67


## 10. Nomes dos hospitais (API do CNES)

O SIH identifica o hospital só pelo código CNES. A API de dados abertos do Ministério da Saúde devolve o nome. Como no OSRM, o resultado fica salvo e a API só é consultada para códigos que ainda não estão no arquivo.

In [15]:
CNES_API = "https://apidadosabertos.saude.gov.br/cnes/estabelecimentos/"

def nome_hospital(cnes):
    try:
        with urllib.request.urlopen(CNES_API + cnes, timeout=30) as r:
            resp = json.load(r)
        return resp.get("nome_fantasia") or resp.get("nome_razao_social")
    except Exception:
        return None

arquivo = PROCESSED / "hospitais_rn.csv"
hospitais = (pd.read_csv(arquivo, dtype={"cnes": str}) if arquivo.exists()
             else pd.DataFrame(columns=["cnes", "nome_hospital"]))
faltam = sorted(set(df["CNES"]) - set(hospitais["cnes"]))
if faltam:
    novos = pd.DataFrame({"cnes": faltam, "nome_hospital": [nome_hospital(c) for c in faltam]})
    hospitais = pd.concat([hospitais, novos], ignore_index=True)
    hospitais.to_csv(arquivo, index=False)

print(f"Hospitais: {len(hospitais)}, sem nome na API: {hospitais['nome_hospital'].isna().sum()}")

Hospitais: 130, sem nome na API: 0
